# 02 · ROI, frames and features (M2 gate + M3)
1. **Confirm the workspace crop (ROI)** on real frames. Edit the box until it covers the work area, then put
   it in `procedures/impact_reassembly_A/roi.yaml` (set `confirmed: true`) and commit, or paste it back.
2. Extract DINOv2 features (ROI and full frame) and resample the shipped VideoMAEv2 features onto the 2 fps grid.
   Everything is cached on Drive by config hash and skips finished executions.
3. Check the temporal alignment of shipped features vs our frames.

## Setup
Mounts Drive, clones or updates the repo, installs it and sets the `SOPVERIFY_*` paths. Safe to re-run after a runtime reset.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%%bash
set -e
REPO=/content/SOP-Verification
BRANCH=claude/nifty-mccarthy-5suut3   # switch to main once the branch is merged
if [ -d $REPO/.git ]; then
  git -C $REPO fetch -q origin $BRANCH && git -C $REPO checkout -q $BRANCH && git -C $REPO pull -q --ff-only origin $BRANCH
else
  git clone -q -b $BRANCH https://github.com/RohanSada/SOP-Verification.git $REPO
fi
pip install -q -e "$REPO[app,dev]"
git -C $REPO log --oneline -1

In [ ]:
import os, sys
sys.path.insert(0, "/content/SOP-Verification/src")   # editable install without a runtime restart
os.chdir("/content/SOP-Verification")
from sopverify.colab import setup_env, show
ROOT = "/content/drive/MyDrive/sopverify"
ENV = setup_env(ROOT)
try:  # optional Hugging Face token (Colab secret named HF_TOKEN), in case the dataset is gated
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
ENV

### 0. Expertise labels 📋
The only metadata file in the release is `NASA_TLX_anonymized.xlsx`. This prints its sheets so we can see whether it says who is an expert. Takes a few seconds.

In [ ]:
import glob, pandas as pd
from sopverify.data.impact.catalog import Catalog
for p in sorted(glob.glob(os.environ["SOPVERIFY_DATA_ROOT"] + "/metadata/*.xls*")):
    for name, df in pd.read_excel(p, sheet_name=None).items():
        print(f"== {p.split('/')[-1]} | sheet {name!r} | {df.shape[0]} rows x {df.shape[1]} columns")
        print("columns:", list(df.columns))
        print(df.head(20).to_string(), "
")
print("expertise parsed:", Catalog.from_paths(os.environ["SOPVERIFY_DATA_ROOT"], None).expertise)

### 1. ROI preview

In [ ]:
from sopverify.data.impact.catalog import Catalog
from sopverify.procedure.loader import load_procedure
from sopverify.video.decode import read_frames_at
from sopverify.video.roi import overlay_box
cat = Catalog.from_paths(os.environ["SOPVERIFY_DATA_ROOT"], None)
proc = load_procedure("procedures/impact_reassembly_A")
BOX = proc.roi.box          # try other boxes here, e.g. [420, 280, 520, 400]
from PIL import Image
ids = cat.execution_ids()
for i, (e, t) in enumerate([(ids[0], 20), (ids[len(ids) // 2], 120), (ids[-1], 200)]):
    ref = cat.ref(e)
    fr = read_frames_at(ref.video_path, [min(t, ref.duration_s - 1)])[0]
    img = overlay_box(fr, BOX)
    show(img, f"{e} t={t}s  box={BOX}")
    Image.fromarray(img).save(f"{os.environ['SOPVERIFY_EXPORTS']}/roi_preview_{i}.jpg", quality=85)
print("saved roi_preview_0..2.jpg in MyDrive/sopverify/exports/")

### 2. Features (resumable). Target: DINOv2-base < 2 min per 20-min video on A100, excluding decode.

In [ ]:
!sopverify features extract --crop roi

In [ ]:
!sopverify features extract --crop full

In [ ]:
!sopverify features extract --extractor shipped_videomaev2

### 3. Coverage and timing 📋

In [ ]:
import time, numpy as np
from sopverify.config import load_config
from sopverify.features.provider import FeatureProvider
base = load_config()
variants = {"dinov2_roi": {"features.crop": "roi"}, "dinov2_full": {"features.crop": "full"},
            "videomae": {"features.extractor": "shipped_videomaev2"}}
for name, ov in variants.items():
    fp = FeatureProvider(base.with_overrides(ov), proc.roi.box, proc.roi.frame_size)
    have = sum(fp.has(cat.ref(e)) for e in ids)
    print(f"{name}: {have}/{len(ids)} cached  key={fp.key_hash()}")
# Encoder speed on frames already in memory (excludes decode):
fp = FeatureProvider(base, proc.roi.box, proc.roi.frame_size)
ref = cat.ref(ids[0]); frames = fp.frames(ref, fp.grid(ref))
t0 = time.time(); fp.extractor().encode(np.asarray(frames)); dt = time.time() - t0
print(f"DINOv2 encode: {len(frames)} frames in {dt:.1f}s -> {dt / len(frames) * 2400:.0f}s per 20-min video at 2 fps")

### 4. Alignment check: shipped VideoMAEv2 vs our frame grid 📋
Both streams should change at the same moments; the best lag of their change signals should be ~0 s.

In [ ]:
from sopverify.features.alignment import alignment_report
fa = FeatureProvider(base.with_overrides({"features.crop": "full"}), proc.roi.box)
fb = FeatureProvider(base.with_overrides({"features.extractor": "shipped_videomaev2"}), proc.roi.box)
rep = []
for e in ids[:10]:
    r = alignment_report(fa.get(cat.ref(e))[1], fb.get(cat.ref(e))[1], base.video.sample_fps)
    rep.append({"execution": e, **r}); print(e, r)
import pandas as pd; print(pd.DataFrame(rep)[["best_lag_s", "peak_corr"]].describe())